In [12]:
import torch
import os 
import json
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR100
from torchvision import transforms, models
%cd /home/devops/data/DL4CV/DL4CV/A1/Retrain
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# TIỀN XỬ LÝ DỮ LIỆU & AUGMENTATION
# ==========================================
transform_train = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomCrop(224, padding=4),
    transforms.RandomHorizontalFlip(),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    ),

    transforms.RandomErasing(
        p=0.25,
        scale=(0.02, 0.15),
        ratio=(0.3, 3.3)
    )
])
# Chuẩn hóa theo Mean và Std của bộ dữ liệu CIFAR-100
# transform_train = transforms.Compose([
#     #transforms.Resize(224),  
#     transforms.RandomCrop(32, padding=4),
#     transforms.RandomHorizontalFlip(),
#     transforms.ToTensor(),
#     transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
# ])

transform_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])
# 4. Khởi tạo Dataset chuẩn
raw_train_dataset  = CIFAR100(root="data", train=True, download=True, transform=transform_train)
raw_val_dataset    = CIFAR100(root="data", train=True, download=False, transform=transform_test)
final_test_dataset = CIFAR100(root="data", train=False, download=True, transform=transform_test)

# 5. Chia Train (90%) - Val (10%) từ 50,000 mẫu
total_samples = len(raw_train_dataset)
train_size = int(0.9 * total_samples)
val_size = total_samples - train_size

generator = torch.Generator().manual_seed(42)
permuted_indices = torch.randperm(total_samples, generator=generator).tolist()

train_indices = permuted_indices[:train_size]
val_indices   = permuted_indices[train_size:]

TRAIN_SUBSET = Subset(raw_train_dataset, train_indices)
VAL_SUBSET   = Subset(raw_val_dataset, val_indices)

# 6. DataLoader với num_workers=0 để tránh lỗi Shared Memory (shm)
BATCH_SIZE = 64
train_loader = DataLoader(TRAIN_SUBSET, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader   = DataLoader(VAL_SUBSET, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(final_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

/home/devops/data/DL4CV/DL4CV/A1/Retrain


In [13]:
# Load ResNet18 pretrained on ImageNet
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# Replace the classification layer: 1000 classes -> 100 classes
model.fc = nn.Linear(model.fc.in_features, 100)

print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [14]:
from count_parameters import count_parameters
params = count_parameters(model)
print(f"Total parameters: {params['total_params']} M")
print(f"Trainable parameters: {params['trainable_params']} M")

Total parameters: 11.227812 M
Trainable parameters: 11.227812 M


In [15]:
# Freeze pretrained backbone
for param in model.parameters():
    param.requires_grad = False

# Enable training for classifier
for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

In [16]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0001,
    weight_decay=0.01
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50,
    eta_min=1e-6
)

criterion = nn.CrossEntropyLoss()

In [10]:
def train_one_epoch_pretrained(
    model,
    train_loader,
    criterion,
    optimizer,
    device,
    strategy
):

    model.train()

    if strategy == "FeatureExtraction":
        # Keep frozen backbone in evaluation mode
        model.eval()
        model.fc.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [18]:
def evaluate(
    model,
    data_loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in data_loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [ ]:
import time

from typeguard import config
import torch
config = {
    "optimizer": "AdamW",
    "learning_rate": 0.0001,
    "weight_decay": 0.01,
    "scheduler": "CosineAnnealingLR",
    "scheduler_T_max": 50,
    "scheduler_eta_min": 1e-6,
    "criterion": "CrossEntropyLoss",
    "pretrained": True,
    "pretrained_dataset": "ImageNet-1K",
    "fine_tuning": True,
    "frozen_backbone": False,
    "input_size": [224, 224],
    "num_classes": 100,
    "epochs": 50,
    "batch_size": 64
}
NUM_EPOCHS = 50
best_val_acc = 0.0
start_time = time.time()

# Initialize logger – giữ cấu trúc như Scratch
logger = ExperimentLogger(
    experiment_id="ResNet18_Pretrained_FineTuning_seed42",
    model_name="ResNet18",
    strategy="Pretrained_FineTuning",
    seed=SEED,
    input_size=[224, 224],
    config=config
)

# Log model complexity
logger.log_model_complexity(model)

try:
    for epoch in range(NUM_EPOCHS):
        epoch_start = time.time()

        # Training
        train_loss, train_acc = train_one_epoch_pretrained(
            model, train_loader, criterion, optimizer, device, strategy="Pretrained_FineTuning"
        )

        # Validation
        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device
        )

        scheduler.step()

        epoch_time = time.time() - epoch_start

        # Log epoch – giữ cấu trúc dictionary như Scratch
        logger.log_epoch(epoch + 1, {
            "epoch": epoch + 1,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "epoch_time_sec": epoch_time,
            "samples_per_sec": len(train_loader.dataset) / epoch_time,
            "memory_allocated_mb": torch.cuda.memory_allocated(device) / 1024**2,
            "memory_reserved_mb": torch.cuda.memory_reserved(device) / 1024**2,
            "peak_memory_allocated_mb": torch.cuda.max_memory_allocated(device) / 1024**2,
            "peak_memory_reserved_mb": torch.cuda.max_memory_reserved(device) / 1024**2
        })

        print(
            f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.2f}%"
        )

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc

            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_acc": val_acc,
                "val_loss": val_loss
            }, "results/checkpoints/best_resnet18_pretrained_finetuning.pth")

            logger.log_best_checkpoint(epoch + 1, val_acc)

    # Load best checkpoint
    checkpoint = torch.load(
        "results/checkpoints/best_resnet18_pretrained_finetuning.pth",
        map_location=device,
        weights_only=True
    )
    model.load_state_dict(checkpoint["model_state_dict"])

    # Test
    test_loss, test_acc = evaluate(
        model, test_loader, criterion, device
    )

    training_time = time.time() - start_time

    # Log test results
    logger.log_test_results({
        "test_loss": test_loss,
        "test_accuracy": test_acc
    })

    # Finish experiment
    logger.finish(
        status="completed",
        extra_metrics={
            "best_val_accuracy": best_val_acc,
            "training_time_sec": training_time
        }
    )

    print(f"\nBest Validation Accuracy: {best_val_acc:.2f}%")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.2f}%")
    print(f"Training Time: {training_time:.2f}s")

except Exception as e:
    logger.log_event("experiment_failed", {
        "error": str(e)
    })
    logger.finish(status="failed")
    raise

Epoch [1/50] Train Loss: 3.8353 | Train Acc: 0.20% | Val Loss: 3.2811 | Val Acc: 0.33%
Epoch [2/50] Train Loss: 3.0925 | Train Acc: 0.37% | Val Loss: 2.6875 | Val Acc: 0.44%
Epoch [3/50] Train Loss: 2.6613 | Train Acc: 0.43% | Val Loss: 2.3525 | Val Acc: 0.48%
Epoch [4/50] Train Loss: 2.3990 | Train Acc: 0.47% | Val Loss: 2.1586 | Val Acc: 0.50%
Epoch [5/50] Train Loss: 2.2290 | Train Acc: 0.49% | Val Loss: 2.0092 | Val Acc: 0.52%
Epoch [6/50] Train Loss: 2.1059 | Train Acc: 0.50% | Val Loss: 1.9191 | Val Acc: 0.53%
Epoch [7/50] Train Loss: 2.0205 | Train Acc: 0.51% | Val Loss: 1.8449 | Val Acc: 0.54%
Epoch [8/50] Train Loss: 1.9526 | Train Acc: 0.52% | Val Loss: 1.7841 | Val Acc: 0.55%
Epoch [9/50] Train Loss: 1.8913 | Train Acc: 0.53% | Val Loss: 1.7215 | Val Acc: 0.56%
Epoch [10/50] Train Loss: 1.8492 | Train Acc: 0.54% | Val Loss: 1.7090 | Val Acc: 0.56%
Epoch [11/50] Train Loss: 1.8108 | Train Acc: 0.54% | Val Loss: 1.6823 | Val Acc: 0.56%
Epoch [12/50] Train Loss: 1.7788 | Train 

In [ ]:
!python plot_experiment_results.py

Tìm thấy 3 experiment.
Không có training history: ResNet18_Pretrained_Fine-tuning_seed42

========== EXPERIMENT SUMMARY ==========

Experiment: ResNet18_Scratch_seed42
Best Epoch: 47
Best Val Accuracy: 66.96
Test Accuracy: 66.8
Training Time: 1846.45 sec
Peak GPU Memory: 824.83 MB

Experiment: ResNet18_Pretrained_Fine-tuning_seed42
Best Epoch: None
Best Val Accuracy: None
Test Accuracy: None
Training Time: None sec
Peak GPU Memory: None MB

Experiment: ResNet18_Pretrained_FineTuning_seed42
Best Epoch: 38
Best Val Accuracy: 0.605
Test Accuracy: None
Training Time: 5557.776336669922 sec
Peak GPU Memory: None MB

Đã hoàn thành vẽ biểu đồ.
Output directory: /home/devops/data/DL4CV/DL4CV/A1/Retrain/results/plots
